# Token-Reduction Rig — Analysis

Narrative view of the three numbers, off the same loader the dashboard uses (`rig_data.py`). Sources: `rig_calls.jsonl` (live proxy traffic) and `ab_results.jsonl` (A/B divergence). Local Ollama models have no prompt cache, so `cache_read_ratio` is replaced by **input-token totals / savings**.

In [ ]:
import matplotlib.pyplot as plt
from rig_data import load_calls, load_ab, noise_floor

calls = load_calls()
ab = load_ab()
print('proxy calls:', len(calls), '| ab rows:', len(ab))
calls.head()

## 1 — input-token totals & savings per transform
Replaces `cache_read_ratio`: lower total input tokens for the same work = the transform is shrinking context.

In [ ]:
if not calls.empty:
    g = calls.groupby('transform').agg(
        total_input=('input_tokens','sum'),
        avg_input_per_turn=('input_tokens','mean')).reset_index()
    fig, ax = plt.subplots(1, 2, figsize=(11,4))
    ax[0].bar(g['transform'], g['total_input']); ax[0].set_title('total input tokens')
    ax[1].bar(g['transform'], g['avg_input_per_turn']); ax[1].set_title('avg input / turn')
    plt.tight_layout(); plt.show()
    display(g)
else:
    print('no proxy calls yet — run a task through the proxy first')

## 2 — per-turn context growth
Input tokens vs turn depth, per session. A steep line = the context tail is ballooning.

In [ ]:
if not calls.empty:
    fig, ax = plt.subplots(figsize=(9,5))
    for (sess, t), d in calls.sort_values('turn').groupby(['session','transform']):
        ax.plot(d['turn'], d['input_tokens'], marker='o', label=f'{sess[:6]}/{t}')
    ax.set_xlabel('turn'); ax.set_ylabel('input tokens'); ax.set_title('context growth')
    ax.legend(fontsize=8); plt.show()
else:
    print('no proxy calls yet')

## 3 — divergence vs noise floor + token savings
Run `ab_runner.py reqs.jsonl identity` first for the noise floor. A transform at the floor is lossless; above it is a loss (or a bug, for `dedup`).

In [ ]:
if not ab.empty:
    floor = noise_floor(ab)
    agg = ab.groupby('transform').agg(divergence=('divergence','mean'),
                                      saving=('saving_frac','mean')).reset_index()
    fig, ax = plt.subplots(1, 2, figsize=(11,4))
    ax[0].bar(agg['transform'], agg['divergence'])
    ax[0].axhline(floor, ls='--', color='r', label=f'floor {floor:.4f}')
    ax[0].set_title('mean divergence'); ax[0].legend()
    ax[1].bar(agg['transform'], agg['saving']); ax[1].set_title('mean input-token saving')
    plt.tight_layout(); plt.show()
    print(f'noise floor = {floor:.4f}'); display(agg)
else:
    print('no ab_runner results yet')